In [1]:
# xgboost model tests for klarna dataset

In [22]:
import joblib

from __future__ import annotations

import json
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Tuple, Any

import joblib
import numpy as np
import optuna
import pandas as pd
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split

In [25]:

data = json.load(open('train_test_data_without_time_tts.json'))

X_train = data['X_train']
X_test = data['X_test']

X_train = pd.DataFrame(X_train)  
X_test = pd.DataFrame(X_test)

y_train = data['y_train']
y_test = data['y_test']

y_train = pd.Series(y_train)
y_test = pd.Series(y_test)


# Config
RANDOM_STATE = 42
ARTIFACTS_DIR = Path("model_artifacts_xgb")
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Metrics

def ks_statistic(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    Compute Kolmogorov-Smirnov statistic for binary classification.
    """
    df = pd.DataFrame({
        "target": np.asarray(y_true),
        "score": np.asarray(y_score)
    }).sort_values("score", ascending=False).reset_index(drop=True)

    n_bad = (df["target"] == 1).sum()
    n_good = (df["target"] == 0).sum()

    if n_bad == 0 or n_good == 0:
        return 0.0

    df["cum_bad"] = (df["target"] == 1).cumsum() / n_bad
    df["cum_good"] = (df["target"] == 0).cumsum() / n_good

    return float(np.max(np.abs(df["cum_bad"] - df["cum_good"])))


def evaluate_at_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float
) -> Dict[str, float]:

    y_pred = (y_prob >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "ks": float(ks_statistic(y_true, y_prob)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    }


def find_best_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    min_recall: float = 0.70,
    threshold_grid: np.ndarray | None = None,
) -> Dict[str, float]:

    if threshold_grid is None:
        threshold_grid = np.arange(0.05, 0.95, 0.01)

    candidates = []

    for threshold in threshold_grid:
        metrics = evaluate_at_threshold(y_true, y_prob, threshold)
        candidates.append(metrics)

    valid_candidates = [m for m in candidates if m["recall"] >= min_recall]

    if valid_candidates:
        best = max(valid_candidates, key=lambda x: (x["precision"], x["recall"]))
    else:
        best = max(candidates, key=lambda x: (x["recall"], x["precision"]))

    return best

# validation

def validate_training_data(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series
) -> None:
   
    if not isinstance(X_train, pd.DataFrame):
        raise TypeError("X_train must be a pandas DataFrame.")
    if not isinstance(X_test, pd.DataFrame):
        raise TypeError("X_test must be a pandas DataFrame.")

    y_train = pd.Series(y_train)
    y_test = pd.Series(y_test)

    if X_train.empty or X_test.empty:
        raise ValueError("X_train/X_test cannot be empty.")

    if len(X_train) != len(y_train):
        raise ValueError("X_train and y_train have different lengths.")

    if len(X_test) != len(y_test):
        raise ValueError("X_test and y_test have different lengths.")

    if list(X_train.columns) != list(X_test.columns):
        raise ValueError("X_train and X_test must have identical columns in same order.")

    non_numeric_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric columns found. Encode them before XGBoost: {non_numeric_cols}"
        )

    unique_target_values = sorted(pd.Series(y_train).dropna().unique().tolist())
    if not set(unique_target_values).issubset({0, 1}):
        raise ValueError(f"Target must be binary 0/1. Found values: {unique_target_values}")

# additional

def compute_scale_pos_weight(y: pd.Series) -> float:
    y = pd.Series(y).astype(int)
    n_negative = (y == 0).sum()
    n_positive = (y == 1).sum()

    if n_positive == 0:
        return 1.0

    return float(n_negative / n_positive)


@dataclass
class ModelArtifacts:
    model_type: str
    best_params: Dict[str, Any]
    validation_threshold: float
    validation_metrics: Dict[str, float]
    test_metrics: Dict[str, float]
    scale_pos_weight: float
    feature_names: list[str]
    best_iteration: int | None


# Optuna

def create_objective(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    scale_pos_weight: float,
):
    def objective(trial: optuna.Trial) -> float:
        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,

            "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 6),
            "min_child_weight": trial.suggest_int("min_child_weight", 10, 80),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            "scale_pos_weight": scale_pos_weight,
        }

        model = XGBClassifier(**params)

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )

        y_val_prob = model.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, y_val_prob)
        val_ks = ks_statistic(y_val, y_val_prob)

        score = 0.7 * val_auc + 0.3 * val_ks
        return float(score)

    return objective



# training function

def train_xgboost(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    validation_size: float = 0.20,
    n_trials: int = 40,
    min_recall_for_threshold: float = 0.70,
) -> Tuple[XGBClassifier, Dict[str, Any], pd.DataFrame]:
    
    validate_training_data(X_train, X_test, y_train, y_test)

    y_train = pd.Series(y_train).astype(int)
    y_test = pd.Series(y_test).astype(int)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train,
        y_train,
        test_size=validation_size,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )

    
    # Imbalance handling
    
    scale_pos_weight = compute_scale_pos_weight(y_tr)
    print(f"scale_pos_weight: {scale_pos_weight:.4f}")

   
    study = optuna.create_study(direction="maximize", study_name="klarna_xgboost")

    study.optimize(
        create_objective(
            X_tr=X_tr,
            y_tr=y_tr,
            X_val=X_val,
            y_val=y_val,
            scale_pos_weight=scale_pos_weight,
        ),
        n_trials=n_trials,
        show_progress_bar=False,
    )

    best_params = study.best_params.copy()
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "scale_pos_weight": scale_pos_weight,
    })

    print("\nBest Optuna params:")
    print(best_params)

    model = XGBClassifier(**best_params)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    y_val_prob = model.predict_proba(X_val)[:, 1]
    best_threshold_metrics = find_best_threshold(
        y_true=y_val,
        y_prob=y_val_prob,
        min_recall=min_recall_for_threshold,
    )
    best_threshold = best_threshold_metrics["threshold"]

    print("\nValidation threshold selection:")
    print(best_threshold_metrics)



    y_test_prob = model.predict_proba(X_test)[:, 1]
    test_metrics = evaluate_at_threshold(
        y_true=y_test,
        y_prob=y_test_prob,
        threshold=best_threshold,
    )

    y_test_pred = (y_test_prob >= best_threshold).astype(int)

    print("\nTest metrics:")
    print(test_metrics)

    print("\nConfusion matrix on test:")
    print(confusion_matrix(y_test, y_test_pred))

    print("\nClassification report on test:")
    print(classification_report(y_test, y_test_pred, digits=4))

    feature_importance = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    best_iteration = getattr(model, "best_iteration", None)

    artifacts = ModelArtifacts(
        model_type="XGBoost",
        best_params=best_params,
        validation_threshold=float(best_threshold),
        validation_metrics=best_threshold_metrics,
        test_metrics=test_metrics,
        scale_pos_weight=float(scale_pos_weight),
        feature_names=list(X_train.columns),
        best_iteration=None if best_iteration is None else int(best_iteration),
    )

    joblib.dump(model, ARTIFACTS_DIR / "xgboost_model.pkl")
    
    feature_importance.to_csv(ARTIFACTS_DIR / "feature_importance.csv", index=False)

    with open(ARTIFACTS_DIR / "model_metadata.json", "w") as f:
        json.dump(asdict(artifacts), f, indent=4)

    print(f"\nArtifacts saved to: {ARTIFACTS_DIR.resolve()}")

    return model, asdict(artifacts), feature_importance

model, metadata, feature_importance = train_xgboost(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    validation_size=0.20,
    n_trials=40,
    min_recall_for_threshold=0.70,
)

print("\nTop 20 important features:")
print(feature_importance.head(20))

[I 2026-03-09 16:53:44,935] A new study created in memory with name: klarna_xgboost


scale_pos_weight: 17.3500


[I 2026-03-09 16:53:46,869] Trial 0 finished with value: 0.5612508393819039 and parameters: {'n_estimators': 609, 'learning_rate': 0.018022942419166226, 'max_depth': 6, 'min_child_weight': 63, 'subsample': 0.7433692752683594, 'colsample_bytree': 0.8135335006189572, 'gamma': 4.770917642603415, 'reg_alpha': 1.1641245439773649, 'reg_lambda': 4.050144523328581}. Best is trial 0 with value: 0.5612508393819039.
[I 2026-03-09 16:53:49,206] Trial 1 finished with value: 0.5385133684019237 and parameters: {'n_estimators': 895, 'learning_rate': 0.08125770807987652, 'max_depth': 6, 'min_child_weight': 37, 'subsample': 0.9809922626113254, 'colsample_bytree': 0.7985578615092901, 'gamma': 2.538127127589825, 'reg_alpha': 1.4844924644254487, 'reg_lambda': 0.010977093458086314}. Best is trial 0 with value: 0.5612508393819039.
[I 2026-03-09 16:53:50,672] Trial 2 finished with value: 0.567731973089246 and parameters: {'n_estimators': 564, 'learning_rate': 0.015581060472048993, 'max_depth': 4, 'min_child_w


Best Optuna params:
{'n_estimators': 753, 'learning_rate': 0.012409987004192387, 'max_depth': 3, 'min_child_weight': 74, 'subsample': 0.6653026741917801, 'colsample_bytree': 0.962512545052971, 'gamma': 2.5539854539971456, 'reg_alpha': 0.0036915046733556963, 'reg_lambda': 2.119876905135531, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1, 'scale_pos_weight': 17.350012830382344}

Validation threshold selection:
{'threshold': 0.4700000000000001, 'roc_auc': 0.6919246419689979, 'ks': 0.28698734537637843, 'recall': 0.7053388090349076, 'precision': 0.08490915832406377}

Test metrics:
{'threshold': 0.4700000000000001, 'roc_auc': 0.6896343652593959, 'ks': 0.27332798680775894, 'recall': 0.715927750410509, 'precision': 0.08488270222914436}

Confusion matrix on test:
[[11729  9401]
 [  346   872]]

Classification report on test:
              precision    recall  f1-score   support

           0     0.9713    0.5551    0.7065     21130

In [26]:
# predict probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# get optimal threshold from validation
threshold = 0.48

# convert probability -> prediction
y_pred = (y_prob >= threshold).astype(int)

In [27]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Precision:", precision)
print("Recall:", recall)
print("ROC AUC:", roc_auc)

Precision: 0.08609406952965235
Recall: 0.6912972085385879
ROC AUC: 0.6896343652593959
